# Gán nhãn

In [25]:
import pandas as pd
df = pd.read_csv("data.csv")
df

,text
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...
...,...
2006,Khoảnh khắc đàn voi xếp vòng tròn bảo vệ con t...
2007,"Biết bạn gái chống chọi bệnh tật suốt 20 năm, ..."
2008,Bí mật phong thủy trong biệt phủ được xây suốt...
2009,"Vườn hồng rực rỡ, rau trái sum sê trên sân thư..."


In [5]:
# ==== 0) CÀI THƯ VIỆN (chạy 1 lần ngoài script)
# pip install snorkel pandas numpy

# ==== 1) IMPORTS & CẤU HÌNH ====
import re, unicodedata, numpy as np, pandas as pd
try:
    # new-ish API (preferred)
    from snorkel.labeling.model.label_model import LabelModel
    from snorkel.labeling.apply.pandas import PandasLFApplier
    from snorkel.labeling import labeling_function
except Exception:
    try:
        # older API surface
        from snorkel.labeling import LabelModel, PandasLFApplier, labeling_function
    except Exception:
        # fallback: try alternate locations
        from snorkel.labeling.model import LabelModel
        from snorkel.labeling.apply.pandas import PandasLFApplier
        from snorkel.labeling import labeling_function

INPUT_CSV = "data.csv"
OUTPUT_CSV = "data_labeled_snorkel.csv"
TEXT_COL = "text"  # ĐỔI nếu tên cột khác

# Nhãn cho Snorkel (giữ đúng thứ tự dùng xuyên suốt)
ABSTAIN, POS, NEG, NEU = -1, 0, 1, 2

# ==== 2) TIỆN ÍCH TIỀN XỬ LÝ ====
def strip_accents(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.replace("đ","d").replace("Đ","D")
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

def norm(s: str) -> str:
    s = strip_accents(s.lower().strip())
    s = re.sub(r"\s+"," ", s)
    return s

def contains_any(text: str, terms) -> bool:
    t = norm(text)
    for w in terms:
        # biên giới từ đơn giản để tránh dính từ khác
        if re.search(rf"\b{re.escape(norm(w))}\b", t):
            return True
    return False

def has_negation_near(text: str, terms, window: int = 4) -> bool:
    """Phát hiện phủ định gần cụm tích cực/tiêu cực trong cửa sổ ±window từ."""
    NEGATIONS = {"khong","ko","k","kg","chua","chua tung","chua he","chua duoc","chang","chong","phi"}
    toks = norm(text).split()
    if not toks: return False
    txt = " ".join(toks)
    # vị trí từng cụm từ cần xét
    for w in terms:
        pat = re.compile(rf"\b{re.escape(norm(w))}\b")
        for m in pat.finditer(txt):
            # xác định khoảng quanh cụm
            start_tok = len(txt[:m.start()].split())
            end_tok   = start_tok + len(norm(w).split()) - 1
            left = max(0, start_tok - window)
            right = min(len(toks)-1, end_tok + window)
            window_text = " ".join(toks[left:right+1])
            if any(re.search(rf"\b{re.escape(neg)}\b", window_text) for neg in NEGATIONS):
                return True
    return False

# ==== 3) BỘ TỪ KHÓA (mở rộng theo miền tin tức) ====
# POSITIVE (tin tốt / thành tựu / phúc lợi / khởi công - ra mắt)
POS_GENERAL = {
    "tich cuc","kha quan","dang mung","thuan loi","hieu qua","an toan","thanh cong",
    "ky luc","tang truong","phuc hoi","cai thien","vuot muc tieu","nang cap","phat trien",
    "vinh danh","duoc khen thuong","dat giai","giai thuong","xep hang cao","tien phong",
    "mo rong","giam phi","giam gia","uu dai","mien phi","ho tro","tro cap","an sinh",
    "phau thuat thanh cong","cuu tro","cuu song","giai cuu","ban giao","dua vao van hanh",
}
POS_EVENT = {
    "khoi cong","khoi dong","khai truong","khanh thanh","ra mat","ra mat san pham",
    "cong bo","phe duyet","thanh lap","tuyen dung","mo ban","mo cua tro lai",
    "thu nghiem thanh cong","de an duoc thong qua","trien khai","tiep suc","hoc bong",
}
POS_EDU_SCI = {
    "nhom nghien cuu manh","doi moi sang tao","chuyen doi so","dat chuan","duoc cong nhan",
    "hoc bong","do dau","dau bang","tang chi tieu tuyen sinh","dat chuan kiem dinh",
}
POS_HEALTH_SOC = {
    "giam benh","kiem soat dich","xoa ngheo","giam ngheo","tang muc song","tang phuc loi",
    "tang luong","tang phu cap","ho tro nguoi dan","bao hiem chi tra",
}

# NEGATIVE (su co / phap ly / suc khoe / kinh te-xa hoi)
NEG_DISASTER = {
    "chay","chay lon","no","sap","sat lo","ngap","lu lut","bao","dong dat","han han",
    "su co","thiet hai","khung hoang","doi dau","mat mat","nghiem trong","cap bach",
}
NEG_CRIME_LEGAL = {
    "lua dao","gia mao","hang gia","thuoc gia","sua gia","to cao","khoi to","bat tam giam",
    "tac trach","tieu cuc","tham nhung","hoa hong","hoi lo","vi pham","sai pham",
    "xu phat","truy to","xet xu","khieu nai","kien","don kien","tranh chap","be boi",
}
NEG_ECON_SOC = {
    "that nghiep","giam luong","thua lo","no xau","lam phat","roi loan","dinh tre","cham tien do",
    "qua tai","ac mong","bat thuong","bat on","bat cap","gay buc xuc","bat binh dang",
}
NEG_HEALTH = {
    "dich benh","bung phat","benh hiem ngheo","tu vong","chet","ngo doc","bong nang",
    "benh dai","benh so","benh nguy hiem","benh truyen nhiem","benh ung thu",
}
NEG_EVENT = {
    "vu chay","vu viec","vu no","tai nan","tien an","tien su","dot kich","ban chet",
    "trom cap","cuop giat","hiep dam","bau vat bi danh cap","gia do","lam gia giay to",
}

# NEUTRAL (tường thuật / cập nhật)
NEU_INFO = {
    "cap nhat","thong bao","huong dan","thong tin chi tiet","bao cao","thong ke","so lieu",
    "lich thi","de thi","dap an","tuyen sinh","phuong an","ke hoach","lich trinh","quy dinh",
    "van ban","du thao","de an","nghi quyet","toan canh","tong quan","tu lieu","ho so",
    "danh sach","top","xep hang","ai la","co phai","vi sao","bao nhieu","la gi",
}
NEU_FORMAT = {
    "anh","video","clip","podcast","livestream","ban tin","tuong thuat","phong su anh"
}

# Gom nhóm để tiện tạo nhiều LF
POS_GROUPS = [POS_GENERAL, POS_EVENT, POS_EDU_SCI, POS_HEALTH_SOC]
NEG_GROUPS = [NEG_DISASTER, NEG_CRIME_LEGAL, NEG_ECON_SOC, NEG_HEALTH, NEG_EVENT]
NEU_GROUPS = [NEU_INFO, NEU_FORMAT]

# ==== 4) TẠO LABELING FUNCTIONS TỪ TỪ KHÓA ====
def make_kw_lf(name: str, terms: set, label: int):
    @labeling_function(name=name)
    def _lf(x):
        return label if contains_any(x.text, terms) else ABSTAIN
    return _lf

LFs = []
# Positive LFs
for i, terms in enumerate(POS_GROUPS, 1):
    LFs.append(make_kw_lf(f"lf_pos_kw_{i}", terms, POS))

# Negative LFs
for i, terms in enumerate(NEG_GROUPS, 1):
    LFs.append(make_kw_lf(f"lf_neg_kw_{i}", terms, NEG))

# Neutral LFs
for i, terms in enumerate(NEU_GROUPS, 1):
    LFs.append(make_kw_lf(f"lf_neu_kw_{i}", terms, NEU))

# Phủ định gần cụm POS => NEG
@labeling_function()
def lf_negation_flip_pos(x):
    return NEG if has_negation_near(x.text, set().union(*POS_GROUPS)) else ABSTAIN

# (tùy) Phủ định gần cụm NEG => POS (ít gặp trong báo chí, nhưng vẫn có)
@labeling_function()
def lf_negation_flip_neg(x):
    return POS if has_negation_near(x.text, set().union(*NEG_GROUPS)) else ABSTAIN

# Tiêu đề/câu hỏi nghi vấn → thường NEU (trừ khi đã trúng POS/NEG ở LFs khác)
@labeling_function()
def lf_question_neu(x):
    t = norm(x.text)
    return NEU if ("?" in x.text or t.startswith(("ai ","co phai ","vi sao ","tai sao ","bao nhieu ","la gi "))) else ABSTAIN

LFs.extend([lf_negation_flip_pos, lf_negation_flip_neg, lf_question_neu])

# ==== 5) ĐỌC DỮ LIỆU ====
df = pd.read_csv(INPUT_CSV)
assert TEXT_COL in df.columns, f"Khong tim thay cot '{TEXT_COL}' trong {INPUT_CSV}"
df = df[[TEXT_COL]].rename(columns={TEXT_COL: "text"})

# ==== 6) ÁP DỤNG LFs & HUẤN LUYỆN LABELMODEL ====
applier = PandasLFApplier(lfs=LFs)
L = applier.apply(df=df)

label_model = LabelModel(cardinality=3, verbose=False)
label_model.fit(L_train=L, n_epochs=500, log_freq=100, seed=7)

# ==== 7) DỰ ĐOÁN & LƯU KẾT QUẢ ====
probs = label_model.predict_proba(L)  # cột theo thứ tự [POS, NEG, NEU]
preds = label_model.predict(L)

out = df.copy()
out[["p_pos","p_neg","p_neu"]] = probs
out["pred_label"] = np.where(preds==POS,"positive", np.where(preds==NEG,"negative","neutral"))
out["confidence"] = out[["p_pos","p_neg","p_neu"]].max(axis=1)
out.to_csv(OUTPUT_CSV, index=False)

print("Phân bố nhãn dự đoán:")
print(out["pred_label"].value_counts(dropna=False))
print(f"\nDa luu: {OUTPUT_CSV}")
print("Goi y review tay: so dong co confidence < 0.5 =", (out["confidence"] < 0.5).sum())

100%|██████████| 500/500 [00:00<00:00, 546.44epoch/s]


Phân bố nhãn dự đoán:
pred_label
neutral     833
negative    746
positive    432
Name: count, dtype: int64

Da luu: data_labeled_snorkel.csv
Goi y review tay: so dong co confidence < 0.5 = 1688


In [6]:
import pandas as pd
df = pd.read_csv("data_labeled_snorkel.csv")
df

,text,p_pos,p_neg,p_neu,pred_label,confidence
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,0.585788,0.140232,0.273980,positive,0.585788
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,0.731767,0.086567,0.181665,positive,0.731767
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,0.319372,0.327784,0.352844,neutral,0.352844
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,0.276561,0.371214,0.352225,negative,0.371214
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,0.333333,0.333333,0.333333,neutral,0.333333
...,...,...,...,...,...,...
2006,Khoảnh khắc đàn voi xếp vòng tròn bảo vệ con t...,0.288214,0.397262,0.314524,negative,0.397262
2007,"Biết bạn gái chống chọi bệnh tật suốt 20 năm, ...",0.333333,0.333333,0.333333,neutral,0.333333
2008,Bí mật phong thủy trong biệt phủ được xây suốt...,0.333333,0.333333,0.333333,neutral,0.333333
2009,"Vườn hồng rực rỡ, rau trái sum sê trên sân thư...",0.313203,0.379658,0.307140,negative,0.379658


In [ ]:
# -*- coding: utf-8 -*-
# Crawl keywords POS/NEG từ VietNamNet theo dải ngày & cate, sau đó gán nhãn data.csv
# Yêu cầu: requests, beautifulsoup4, pandas, numpy, snorkel, tqdm

import re, time, json, unicodedata, sys
from collections import defaultdict
from datetime import datetime, timedelta
from typing import List, Dict, Set
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from tqdm import tqdm

# ================== CẤU HÌNH ==================
START_DATE = "12/04/2025"   # dd/mm/yyyy
END_DATE   = "18/10/2025"   # dd/mm/yyyy

BASE = "https://vietnamnet.vn/tin-tuc-24h"
CATE_CODES = [
    "000006","00000R","00000E","000001","00MK8V","000007","00000U","000003",
    "00000B","000008","00000W","000005","000009","000002","00M7F7","00000T","000004"
]

INPUT_CSV  = "data.csv"                   # file dữ liệu của bạn
OUTPUT_CSV = "data_labeled_snorkel.csv"   # file xuất kết quả

HEADERS = {"User-Agent": "Mozilla/5.0 (keyword-miner/1.0)"}
ARTICLE_HREF_RE = re.compile(r"/[a-z0-9\-]+-\d+\.html$")  # ...-2454143.html

# ================== TIỆN ÍCH CHUẨN HÓA ==================
def strip_accents(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.replace("đ","d").replace("Đ","D")
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

def norm(s: str) -> str:
    s = strip_accents(s.lower().strip())
    s = re.sub(r"\s+"," ", s)
    return s

def contains_any(text: str, terms: Set[str]) -> bool:
    t = norm(text)
    for w in terms:
        if re.search(rf"\b{re.escape(norm(w))}\b", t):
            return True
    return False

def has_negation_near(text: str, terms: Set[str], window: int = 4) -> bool:
    NEGATIONS = {"khong","ko","k","kg","chua","chua tung","chua he","chua duoc","chang"}
    toks = norm(text).split()
    if not toks: return False
    txt = " ".join(toks)
    for w in terms:
        pat = re.compile(rf"\b{re.escape(norm(w))}\b")
        for m in pat.finditer(txt):
            start_tok = len(txt[:m.start()].split())
            end_tok   = start_tok + len(norm(w).split()) - 1
            left = max(0, start_tok - window)
            right = min(len(toks)-1, end_tok + window)
            window_text = " ".join(toks[left:right+1])
            if any(re.search(rf"\b{re.escape(neg)}\b", window_text) for neg in NEGATIONS):
                return True
    return False

# ================== SEED LEXICON (mở rộng theo báo chí) ==================
SEED_POS = {
    "tich cuc","kha quan","dang mung","thuan loi","hieu qua","an toan","thanh cong",
    "ky luc","tang truong","phuc hoi","cai thien","vuot muc tieu","nang cap","phat trien",
    "vinh danh","khen thuong","dat giai","giai thuong","xep hang cao","tien phong",
    "mo rong","giam phi","giam gia","uu dai","mien phi","ho tro","tro cap","an sinh",
    "ban giao","dua vao van hanh","ky ket","hop tac","hop dong lon","goi von","ipo thanh cong",
    "loi nhuan ky luc","doanh thu ky luc","khoi sac","phuc hoi manh","trung thau","giai ngan",
    "khoi cong","khoi dong","khai truong","khanh thanh","ra mat","cong bo","phe duyet","thanh lap",
    "mo ban","mo cua tro lai","thu nghiem thanh cong","de an duoc thong qua","trien khai",
    "tiep suc","hoc bong","nhom nghien cuu manh","doi moi sang tao","chuyen doi so",
    "dat chuan","duoc cong nhan","do dau","dau bang","tang chi tieu tuyen sinh","dat chuan kiem dinh",
    "phau thuat thanh cong","ghep tang thanh cong","khoi benh","xuat vien","kiem soat dich",
    "xoa ngheo","giam ngheo","tang muc song","tang phuc loi","tang luong","tang phu cap",
    "bao hiem chi tra","cuu tro","cuu song","giai cuu",
    "gianh chien thang","chien thang","vo dich","len ngoi","pha ky luc","vao chung ket",
    "thang dam","thang sat nut","thang nhe","thang quan",
    "dong khach","chay ve","doat giai","giai thuong quoc te","ra mat phim","ra mat mv",
    "mo ban du an","ban giao nha","ban giao so","giam lai suat","phe duyet quy hoach",
    "ban cap nhat","ban va","phat song","phu song","ra mat ung dung"
}

SEED_NEG = {
    "chay","chay lon","no","sap","sat lo","ngap","lu lut","bao","dong dat","han han","su co","thiet hai",
    "khung hoang","bat on","bat thuong","cap bach","dot bien xau",
    "lua dao","gia mao","hang gia","thuoc gia","sua gia","to cao","khoi to","bat tam giam",
    "tac trach","tieu cuc","tham nhung","hoi lo","vi pham","sai pham","xu phat","truy to","xet xu",
    "khieu nai","kien","don kien","tranh chap","be boi","truy na","an mang",
    "that nghiep","giam luong","thua lo","no xau","lam phat","roi loan","dinh tre","cham tien do",
    "qua tai","giam manh","giam sau","suy giam","suy thoai","khong dat","tut doc",
    "dich benh","bung phat","o dich","benh hiem ngheo","ngo doc","tu vong","chet","sot xuat huyet",
    "sap mang","sap he thong","lo hong","lo hong bao mat","ro ri du lieu","tan cong mang","ransomware","bi hack",
    "tin gia","mao danh","lua dao truc tuyen","spam",
    "trieu hoi xe","tai nan","lat xe","dieu tra nguyen nhan",
    "chat chem","ket xe","qua tai khach","tai nan du lich",
    "lo de","gian lan thi cu","bao luc hoc duong",
    "thua dam","bi loai","dut mach thang","chan thuong nang",
    "thoi gia","sot dat","bong bong","du an ma","ket phap ly","cham cap so","cuong che","tranh chap dat dai",
    "lum xum","be boi","bi chi trich","tay chay"
}

NEU_HINTS = {
    "cap nhat","thong bao","huong dan","bao cao","thong ke","so lieu",
    "lich thi","de thi","dap an","tuyen sinh","ke hoach","lich trinh","quy dinh",
    "danh sach","top","xep hang","ai la","co phai","vi sao","bao nhieu","la gi","hoi dap"
}

# ================== CRAWL THEO DẢI NGÀY & CATE ==================
def iter_dates(start_str: str, end_str: str):
    s = datetime.strptime(start_str, "%d/%m/%Y")
    e = datetime.strptime(end_str, "%d/%m/%Y")
    d = s
    while d <= e:
        yield d.strftime("%d/%m/%Y")
        d += timedelta(days=1)

def fetch_titles_for_cate(cate: str, bydate: str) -> List[str]:
    url = f"{BASE}?bydate={requests.utils.quote(bydate, safe='')}&cate={cate}"
    for attempt in range(3):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            titles = set()
            for a in soup.find_all("a", href=True):
                href = a["href"]
                if ARTICLE_HREF_RE.search(href):
                    text = a.get_text(strip=True)
                    if text:
                        titles.add(text)
            return list(titles)
        except Exception:
            if attempt == 2:
                return []
            time.sleep(1.0 + attempt * 1.0)
    return []

def collect_keywords_range(start_date: str, end_date: str, cate_codes: List[str]):
    pos_by_cate = {c: set() for c in cate_codes}
    neg_by_cate = {c: set() for c in cate_codes}
    total_titles = 0

    for day in tqdm(list(iter_dates(start_date, end_date)), desc="Days"):
        bydate = f"{day}-{day}"
        for cate in tqdm(cate_codes, desc=f"cate", leave=False):
            titles = fetch_titles_for_cate(cate, bydate)
            total_titles += len(titles)
            # trích keyword xuất hiện
            for t in titles:
                tnorm = norm(t)
                for w in SEED_POS:
                    if re.search(rf"\b{re.escape(norm(w))}\b", tnorm):
                        pos_by_cate[cate].add(w)
                for w in SEED_NEG:
                    if re.search(rf"\b{re.escape(norm(w))}\b", tnorm):
                        neg_by_cate[cate].add(w)
            time.sleep(0.2)  # lịch sự
    return {
        "pos_by_cate": {k: sorted(v) for k, v in pos_by_cate.items()},
        "neg_by_cate": {k: sorted(v) for k, v in neg_by_cate.items()},
        "meta": {"start": start_date, "end": end_date, "total_titles_seen": total_titles}
    }

print("==> Bắt đầu crawl từ", START_DATE, "đến", END_DATE)
kw = collect_keywords_range(START_DATE, END_DATE, CATE_CODES)

with open("vnnet_keywords_range.json", "w", encoding="utf-8") as f:
    json.dump(kw, f, ensure_ascii=False, indent=2)
print("Đã lưu vnnet_keywords_range.json")

# Hợp nhất keyword POS/NEG thực sự gặp trong dải ngày
pos_union = sorted(set().union(*[set(v) for v in kw["pos_by_cate"].values()])) if kw["pos_by_cate"] else []
neg_union = sorted(set().union(*[set(v) for v in kw["neg_by_cate"].values()])) if kw["neg_by_cate"] else []
print(f"POS found (union): {len(pos_union)} | NEG found (union): {len(neg_union)}")

# ================== PHẦN GÁN NHÃN ==================
# Thử import Snorkel (robust qua nhiều phiên bản); nếu fail -> fallback rule-based
SNORKEL_OK = True
try:
    from snorkel.labeling.model.label_model import LabelModel
    from snorkel.labeling.apply.pandas import PandasLFApplier
    from snorkel.labeling import labeling_function
except Exception:
    try:
        from snorkel.labeling import LabelModel, PandasLFApplier, labeling_function
    except Exception:
        try:
            from snorkel.labeling.model import LabelModel
            from snorkel.labeling.apply.pandas import PandasLFApplier
            from snorkel.labeling import labeling_function
        except Exception as e:
            print("[WARN] Snorkel import failed -> dùng fallback rule-based. Lý do:", e)
            SNORKEL_OK = False

ABSTAIN, POS, NEG, NEU = -1, 0, 1, 2
POS_TERMS = set(SEED_POS).union(pos_union)
NEG_TERMS = set(SEED_NEG).union(neg_union)

# Đọc dữ liệu cần gán nhãn
df = pd.read_csv(INPUT_CSV)
if "text" not in df.columns:
    raise ValueError("Không thấy cột 'text' trong data.csv. Hãy đổi tên cột văn bản thành 'text' hoặc sửa code.")

# ---- NEU hints (kéo về neutral khi không có polar rõ)
def lf_neu_hint_func(text: str) -> bool:
    return contains_any(text, NEU_HINTS)

if SNORKEL_OK:
    # === Snorkel LFs ===
    @labeling_function()
    def lf_pos_kw(x):
        return POS if contains_any(x.text, POS_TERMS) else ABSTAIN

    @labeling_function()
    def lf_neg_kw(x):
        return NEG if contains_any(x.text, NEG_TERMS) else ABSTAIN

    @labeling_function()
    def lf_negation_flip_pos(x):
        return NEG if has_negation_near(x.text, POS_TERMS) else ABSTAIN

    @labeling_function()
    def lf_negation_flip_neg(x):
        return POS if has_negation_near(x.text, NEG_TERMS) else ABSTAIN

    @labeling_function()
    def lf_neu_fallback(x):
        return NEU if lf_neu_hint_func(x.text) else ABSTAIN

    LFs = [lf_pos_kw, lf_neg_kw, lf_negation_flip_pos, lf_negation_flip_neg, lf_neu_fallback]

    df_ = df[["text"]].copy()
    applier = PandasLFApplier(lfs=LFs)
    L = applier.apply(df=df_)

    label_model = LabelModel(cardinality=3, verbose=False)
    label_model.fit(L_train=L, n_epochs=500, log_freq=100, seed=7)

    probs = label_model.predict_proba(L)  # [POS, NEG, NEU]
    preds = label_model.predict(L)

    out = df.copy()
    out[["p_pos","p_neg","p_neu"]] = probs
    out["pred_label"] = np.where(preds==POS,"positive", np.where(preds==NEG,"negative","neutral"))
    out["confidence"] = out[["p_pos","p_neg","p_neu"]].max(axis=1)
else:
    # === Fallback rule-based (không cần Snorkel) ===
    def score_rule(text: str):
        t = norm(text)
        pos = sum(1 for w in POS_TERMS if re.search(rf"\b{re.escape(norm(w))}\b", t))
        neg = sum(1 for w in NEG_TERMS if re.search(rf"\b{re.escape(norm(w))}\b", t))
        # phủ định gần
        if has_negation_near(text, POS_TERMS): neg += 1
        if has_negation_near(text, NEG_TERMS): pos += 1
        neu = 1 if lf_neu_hint_func(text) else 0
        return {"pos": pos, "neg": neg, "neu": neu}

    scores = df["text"].apply(score_rule)
    out = df.copy()
    out["p_pos"] = scores.apply(lambda s: s["pos"])
    out["p_neg"] = scores.apply(lambda s: s["neg"])
    out["p_neu"] = scores.apply(lambda s: s["neu"])
    # chuẩn hóa pseudo-prob
    sm = (out[["p_pos","p_neg","p_neu"]].sum(axis=1) + 1e-9)
    out[["p_pos","p_neg","p_neu"]] = out[["p_pos","p_neg","p_neu"]].div(sm, axis=0)
    out["pred_label"] = out[["p_pos","p_neg","p_neu"]].idxmax(axis=1).map({"p_pos":"positive","p_neg":"negative","p_neu":"neutral"})
    out["confidence"] = out[["p_pos","p_neg","p_neu"]].max(axis=1)

# Xuất kết quả
out.to_csv(OUTPUT_CSV, index=False)
print("\n==> PHÂN BỐ NHÃN:")
print(out["pred_label"].value_counts(dropna=False))
print("Đã lưu:", OUTPUT_CSV)
print("Gợi ý review tay (confidence < 0.5):", int((out["confidence"] < 0.5).sum()))


In [2]:
# Re-run the full pipeline to read ./data.csv, extract POS/NEG terms present,
# and label with Snorkel LabelModel (fallback to rule-based if Snorkel unavailable).
import pandas as pd, re, json, unicodedata
import numpy as np
from collections import Counter

def strip_accents(s: str) -> str:
    if not isinstance(s, str): 
        return ""
    s = s.replace("đ","d").replace("Đ","D")
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))

def norm(s: str) -> str:
    s = strip_accents(s.lower().strip())
    s = re.sub(r"\s+"," ", s)
    return s

def contains_any(text: str, terms) -> bool:
    t = norm(text)
    for w in terms:
        if re.search(rf"\b{re.escape(norm(w))}\b", t):
            return True
    return False

def has_negation_near(text: str, terms, window: int = 4) -> bool:
    NEGATIONS = {"khong","ko","k","kg","chua","chua tung","chua he","chua duoc","chang"}
    toks = norm(text).split()
    if not toks: return False
    txt = " ".join(toks)
    for w in terms:
        pat = re.compile(rf"\b{re.escape(norm(w))}\b")
        for m in pat.finditer(txt):
            start_tok = len(txt[:m.start()].split())
            end_tok   = start_tok + len(norm(w).split()) - 1
            left = max(0, start_tok - window)
            right = min(len(toks)-1, end_tok + window)
            window_text = " ".join(toks[left:right+1])
            if any(re.search(rf"\b{re.escape(neg)}\b", window_text) for neg in NEGATIONS):
                return True
    return False

# Load CSV
csv_path = "./data.csv"
df = pd.read_csv(csv_path)

# Detect text column
text_col = None
for c in ["text", "title", "headline", "content"]:
    if c in df.columns:
        text_col = c
        break
if text_col is None:
    for c in df.columns:
        if df[c].dtype == "object":
            text_col = c
            break
if text_col is None:
    raise ValueError("Không thấy cột văn bản ('text'/'title'/'headline'/'content').")

df = df[df[text_col].notna()].copy()
df["_text_norm"] = df[text_col].apply(norm)

# Seed lexicons
SEED_POS = {
    "tich cuc","kha quan","dang mung","thuan loi","hieu qua","an toan","thanh cong",
    "ky luc","tang truong","phuc hoi","cai thien","vuot muc tieu","nang cap","phat trien",
    "vinh danh","khen thuong","dat giai","giai thuong","xep hang cao","tien phong",
    "mo rong","giam phi","giam gia","uu dai","mien phi","ho tro","tro cap","an sinh",
    "ban giao","dua vao van hanh","ky ket","hop tac","hop dong lon","goi von","ipo thanh cong",
    "loi nhuan ky luc","doanh thu ky luc","khoi sac","phuc hoi manh","trung thau","giai ngan",
    "khoi cong","khoi dong","khai truong","khanh thanh","ra mat","cong bo","phe duyet","thanh lap",
    "mo ban","mo cua tro lai","thu nghiem thanh cong","de an duoc thong qua","trien khai",
    "tiep suc","hoc bong","nhom nghien cuu manh","doi moi sang tao","chuyen doi so",
    "dat chuan","duoc cong nhan","do dau","dau bang","tang chi tieu tuyen sinh","dat chuan kiem dinh",
    "phau thuat thanh cong","ghep tang thanh cong","khoi benh","xuat vien","kiem soat dich",
    "xoa ngheo","giam ngheo","tang muc song","tang phuc loi","tang luong","tang phu cap",
    "bao hiem chi tra","cuu tro","cuu song","giai cuu",
    "gianh chien thang","chien thang","vo dich","len ngoi","pha ky luc","vao chung ket",
    "thang dam","thang sat nut","thang nhe","thang quan",
    "dong khach","chay ve","doat giai","giai thuong quoc te","ra mat phim","ra mat mv",
    "mo ban du an","ban giao nha","ban giao so","giam lai suat","phe duyet quy hoach",
    "ban cap nhat","phat song","phu song","ra mat ung dung"
}

SEED_NEG = {
    "chay","chay lon","no","sap","sat lo","ngap","lu lut","bao","dong dat","han han","su co","thiet hai",
    "khung hoang","bat on","bat thuong","cap bach","dot bien xau",
    "lua dao","gia mao","hang gia","thuoc gia","sua gia","to cao","khoi to","bat tam giam",
    "tac trach","tieu cuc","tham nhung","hoi lo","vi pham","sai pham","xu phat","truy to","xet xu",
    "khieu nai","kien","don kien","tranh chap","be boi","truy na","an mang",
    "that nghiep","giam luong","thua lo","no xau","lam phat","roi loan","dinh tre","cham tien do",
    "qua tai","giam manh","giam sau","suy giam","suy thoai","khong dat","tut doc",
    "dich benh","bung phat","o dich","benh hiem ngheo","ngo doc","tu vong","chet","sot xuat huyet",
    "sap mang","sap he thong","lo hong","lo hong bao mat","ro ri du lieu","tan cong mang","ransomware","bi hack",
    "tin gia","mao danh","lua dao truc tuyen","spam",
    "trieu hoi xe","tai nan","lat xe","dieu tra nguyen nhan",
    "chat chem","ket xe","qua tai khach","tai nan du lich",
    "lo de","gian lan thi cu","bao luc hoc duong",
    "thua dam","bi loai","dut mach thang","chan thuong nang",
    "thoi gia","sot dat","bong bong","du an ma","ket phap ly","cham cap so","cuong che","tranh chap dat dai",
    "lum xum","be boi","bi chi trich","tay chay"
}

NEU_HINTS = {
    "cap nhat","thong bao","huong dan","bao cao","thong ke","so lieu",
    "lich thi","de thi","dap an","tuyen sinh","ke hoach","lich trinh","quy dinh",
    "danh sach","top","xep hang","ai la","co phai","vi sao","bao nhieu","la gi","hoi dap"
}

# Which terms appear in data?
def find_terms_in_corpus(texts_norm, term_set):
    found = Counter()
    for t in texts_norm:
        for w in term_set:
            if re.search(rf"\b{re.escape(norm(w))}\b", t):
                found[w] += 1
    return found

pos_found = find_terms_in_corpus(df["_text_norm"], SEED_POS)
neg_found = find_terms_in_corpus(df["_text_norm"], SEED_NEG)

pos_terms_sorted = [k for k,_ in pos_found.most_common()]
neg_terms_sorted = [k for k,_ in neg_found.most_common()]

with open("./pos_terms_found.json","w",encoding="utf-8") as f:
    json.dump({"pos_terms": pos_terms_sorted, "counts": pos_found}, f, ensure_ascii=False, indent=2)
with open("./neg_terms_found.json","w",encoding="utf-8") as f:
    json.dump({"neg_terms": neg_terms_sorted, "counts": neg_found}, f, ensure_ascii=False, indent=2)

# Try Snorkel
SNORKEL_OK = True
try:
    from snorkel.labeling.model.label_model import LabelModel
    from snorkel.labeling.apply.pandas import PandasLFApplier
    from snorkel.labeling import labeling_function
except Exception:
    try:
        from snorkel.labeling import LabelModel, PandasLFApplier, labeling_function
    except Exception:
        try:
            from snorkel.labeling.model import LabelModel
            from snorkel.labeling.apply.pandas import PandasLFApplier
            from snorkel.labeling import labeling_function
        except Exception as e:
            SNORKEL_OK = False
            snorkel_err = str(e)

ABSTAIN, POS, NEG, NEU = -1, 0, 1, 2
POS_TERMS = set(pos_terms_sorted) if pos_terms_sorted else SEED_POS
NEG_TERMS = set(neg_terms_sorted) if neg_terms_sorted else SEED_NEG

def lf_neu_hint_func(text: str) -> bool:
    return contains_any(text, NEU_HINTS)

if SNORKEL_OK:
    @labeling_function()
    def lf_pos_kw(x):
        return POS if contains_any(x[text_col], POS_TERMS) else ABSTAIN

    @labeling_function()
    def lf_neg_kw(x):
        return NEG if contains_any(x[text_col], NEG_TERMS) else ABSTAIN

    @labeling_function()
    def lf_negation_flip_pos(x):
        return NEG if has_negation_near(x[text_col], POS_TERMS) else ABSTAIN

    @labeling_function()
    def lf_negation_flip_neg(x):
        return POS if has_negation_near(x[text_col], NEG_TERMS) else ABSTAIN

    @labeling_function()
    def lf_neu_fallback(x):
        return NEU if lf_neu_hint_func(x[text_col]) else ABSTAIN

    LFs = [lf_pos_kw, lf_neg_kw, lf_negation_flip_pos, lf_negation_flip_neg, lf_neu_fallback]

    applier = PandasLFApplier(lfs=LFs)
    L = applier.apply(df=df[[text_col]])

    label_model = LabelModel(cardinality=3, verbose=False)
    label_model.fit(L_train=L, n_epochs=500, log_freq=100, seed=7)

    probs = label_model.predict_proba(L)  # [POS, NEG, NEU]
    preds = label_model.predict(L)

    out = df.copy()
    out[["p_pos","p_neg","p_neu"]] = probs
    out["pred_label"] = np.where(preds==POS,"positive", np.where(preds==NEG,"negative","neutral"))
    out["confidence"] = out[["p_pos","p_neg","p_neu"]].max(axis=1)
    out_path = "./data_labeled_snorkel.csv"
    out.to_csv(out_path, index=False)
    snorkel_status = "Snorkel LabelModel: OK"
else:
    # Fallback: majority-like
    def score_rule(text: str):
        t = norm(text)
        pos = sum(1 for w in POS_TERMS if re.search(rf"\b{re.escape(norm(w))}\b", t))
        neg = sum(1 for w in NEG_TERMS if re.search(rf"\b{re.escape(norm(w))}\b", t))
        if has_negation_near(text, POS_TERMS): neg += 1
        if has_negation_near(text, NEG_TERMS): pos += 1
        neu = 1 if lf_neu_hint_func(text) else 0
        return {"pos":pos, "neg":neg, "neu":neu}

    scores = df[text_col].apply(score_rule)
    out = df.copy()
    out["p_pos"] = scores.apply(lambda s: s["pos"])
    out["p_neg"] = scores.apply(lambda s: s["neg"])
    out["p_neu"] = scores.apply(lambda s: s["neu"])
    sm = (out[["p_pos","p_neg","p_neu"]].sum(axis=1) + 1e-9)
    out[["p_pos","p_neg","p_neu"]] = out[["p_pos","p_neg","p_neu"]].div(sm, axis=0)
    out["pred_label"] = out[["p_pos","p_neg","p_neu"]].idxmax(axis=1).map({"p_pos":"positive","p_neg":"negative","p_neu":"neutral"})
    out["confidence"] = out[["p_pos","p_neg","p_neu"]].max(axis=1)
    out_path = "./data_labeled_snorkel.csv"
    out.to_csv(out_path, index=False)
    snorkel_status = f"Snorkel LabelModel: NOT AVAILABLE; used fallback."

# Show small preview
preview = out[[text_col, "pred_label", "p_pos","p_neg","p_neu","confidence"]].head(20)

preview


100%|██████████| 500/500 [00:00<00:00, 596.22epoch/s] 


,text,pred_label,p_pos,p_neg,p_neu,confidence
0,Từ học bổng thay đổi cuộc đời đến lớp học tiến...,positive,0.394536,0.276980,0.328484,0.394536
1,Trường ĐH Tôn Đức Thắng công bố 4 nhóm nghiên ...,positive,0.394536,0.276980,0.328484,0.394536
2,Đề thi thử tốt nghiệp THPT môn Toán của thành ...,neutral,0.332652,0.303936,0.363412,0.363412
3,Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo v...,neutral,0.333333,0.333333,0.333333,0.333333
4,Người mẹ trăn trở trước giờ ghi 'đơn xanh' ngu...,neutral,0.333333,0.333333,0.333333,0.333333
5,"Tốt nghiệp loại giỏi dù không biết đọc viết, n...",negative,0.294307,0.395404,0.310289,0.395404
6,Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với g...,neutral,0.333333,0.333333,0.333333,0.333333
7,Tổng Bí thư gợi mở miễn phí bữa trưa cho học s...,positive,0.354485,0.334350,0.311165,0.354485
8,Top 10 trường có điểm chuẩn lớp 10 cao nhất TP...,neutral,0.332652,0.303936,0.363412,0.363412
9,"Ai là người vẽ bản đồ tác chiến Xuân Lộc 1975,...",positive,0.392002,0.251444,0.356553,0.392002


In [5]:

df = pd.read_csv("data_labeled_snorkel.csv")
print("Gợi ý review tay (confidence <= 0.6):", int((df["confidence"] <= 0.5).sum()))

Gợi ý review tay (confidence <= 0.6): 1921


# Kiểm tra nhãn được gán

In [21]:
cols = ['text', 'pred_label_revised', 'confidence_revised']
view = df[cols]

In [22]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [34]:
df = pd.read_csv("data_annotated.csv")
print("Gợi ý review tay (confidence <= 0.6):", int((df["confidence_revised"] <= 0.6).sum()))

Gợi ý review tay (confidence <= 0.6): 921


In [35]:
df[df["confidence_revised"] <= 0.6]

,text,pred_label_revised,confidence_revised
0,"Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo viên bị kiểm tra trình độ tiếng Anh — Sở GD-ĐT TPHCM khẳng định việc khảo sát tiếng Anh của giáo viên không phải kiểm tra trình độ cá nhân. Kết quả khảo sát tuyệt đối không được sử dụng cho bất kỳ mục đích nào khác như đánh giá thi đua, xét lương, kỷ luật hay các mục đích cá nhân khác.",neutral,0.6
1,"Người mẹ trăn trở trước giờ ghi 'đơn xanh' nguyện vọng thi lớp 10 cho con — Trước sức nóng của kỳ thi vào lớp 10 Hà Nội, chị Nguyễn Thị Hoa (Hà Nội) nhiều ngày qua đau đầu trăn trở, “đặt bút lên lại bỏ xuống” trước những quyết định đăng ký nguyện vọng chọn trường cho con.",neutral,0.6
2,"Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với giám đốc Cerberus Esports — Ngoài thể hiện sự tự tin và khả năng trình bày lưu loát bằng tiếng Anh, học sinh tiểu học tại BRIS còn khiến giám đốc Cerberus Esports bất ngờ trước loạt câu hỏi đầy tính phản biện và những ý tưởng kinh doanh cực kỳ sáng tạo.",neutral,0.6
3,"Xây nhà ở nhiều năm trên đất khai hoang có được cấp sổ đỏ? — Thửa đất do ông bà tự khai phá năm 1980 để làm vườn. Năm 2007, ông bà cho con cháu để làm nhà và gia đình đã quản lý ổn định từ đó đến nay. Đất phù hợp quy hoạch nhưng chưa được đăng ký, cấp giấy chứng nhận quyền sử dụng đất lần đầu.",neutral,0.6
4,"Từ chiến sĩ nặng 40kg đến người vẽ bản đồ tác chiến vào 'cánh cửa thép' Xuân Lộc — Cựu chiến binh Đàm Duy Thiên, nguyên trinh sát, người làm công tác bản đồ thuộc Trung đoàn 266, Sư đoàn 341-Sông Lam chia sẻ về công tác vẽ bản đồ tác chiến tấn công vào “cánh cửa thép” Xuân Lộc.",neutral,0.6
5,"Thủy thủ tàu ngầm đọc sách giữa lòng đại dương sâu thẳm — Giữa lòng đại dương sâu thẳm, nơi gần như cách biệt hoàn toàn với thế giới bên ngoài, những thủy thủ tàu ngầm Lữ đoàn 189 duy trì một nét đẹp văn hóa đáng trân trọng.",neutral,0.6
6,"Tổng Bí thư tri ân các tướng lĩnh, anh hùng lực lượng vũ trang — Tổng Bí thư Tô Lâm nhấn mạnh, đại thắng mùa Xuân 1975 mãi mãi là niềm tự hào, là mốc son chói lọi nhất trong lịch sử của dân tộc, một biểu tượng sáng ngời của chủ nghĩa anh hùng cách mạng.",neutral,0.6
7,"Lý do hợp nhất Thái Bình với Hưng Yên, lấy tên tỉnh mới là Hưng Yên — Tên gọi “tỉnh Hưng Yên” sau khi hợp nhất Hưng Yên với Thái Bình là phù hợp, vì địa danh này có bề dày lịch sử, văn hiến và truyền thống cách mạng, xuất hiện từ thời vua Minh Mạng năm 1831.",neutral,0.6
8,"Phu nhân Thủ tướng chia sẻ về Bắc Bling, nghe quan họ với các nhà ngoại giao nữ — Sáng nay, tại tỉnh Bắc Ninh, bà Lê Thị Bích Trân, Phu nhân Thủ tướng Phạm Minh Chính gặp mặt, giao lưu với Nhóm Phụ nữ Cộng đồng ASEAN tại Hà Nội.",neutral,0.6
9,"'Chúng tôi xung trận không phải để thành anh hùng' — ""Chúng tôi còn sống qua chiến tranh là may mắn và cũng là sứ mệnh, sống để tiếp tục cống hiến cho Tổ quốc thay cả phần những đồng đội đã hy sinh"" - nữ Anh hùng Phan Thị Ngọc Tươi chia sẻ.",neutral,0.6


In [23]:
from IPython.display import display
display(view)

,text,pred_label_revised,confidence_revised
0,"Sở Giáo dục TPHCM lên tiếng việc 47.000 giáo viên bị kiểm tra trình độ tiếng Anh — Sở GD-ĐT TPHCM khẳng định việc khảo sát tiếng Anh của giáo viên không phải kiểm tra trình độ cá nhân. Kết quả khảo sát tuyệt đối không được sử dụng cho bất kỳ mục đích nào khác như đánh giá thi đua, xét lương, kỷ luật hay các mục đích cá nhân khác.",neutral,0.600000
1,"Người mẹ trăn trở trước giờ ghi 'đơn xanh' nguyện vọng thi lớp 10 cho con — Trước sức nóng của kỳ thi vào lớp 10 Hà Nội, chị Nguyễn Thị Hoa (Hà Nội) nhiều ngày qua đau đầu trăn trở, “đặt bút lên lại bỏ xuống” trước những quyết định đăng ký nguyện vọng chọn trường cho con.",neutral,0.600000
2,"Màn ‘hỏi xoáy’ bất ngờ của học sinh BRIS với giám đốc Cerberus Esports — Ngoài thể hiện sự tự tin và khả năng trình bày lưu loát bằng tiếng Anh, học sinh tiểu học tại BRIS còn khiến giám đốc Cerberus Esports bất ngờ trước loạt câu hỏi đầy tính phản biện và những ý tưởng kinh doanh cực kỳ sáng tạo.",neutral,0.600000
3,"Xây nhà ở nhiều năm trên đất khai hoang có được cấp sổ đỏ? — Thửa đất do ông bà tự khai phá năm 1980 để làm vườn. Năm 2007, ông bà cho con cháu để làm nhà và gia đình đã quản lý ổn định từ đó đến nay. Đất phù hợp quy hoạch nhưng chưa được đăng ký, cấp giấy chứng nhận quyền sử dụng đất lần đầu.",neutral,0.600000
4,"Từ chiến sĩ nặng 40kg đến người vẽ bản đồ tác chiến vào 'cánh cửa thép' Xuân Lộc — Cựu chiến binh Đàm Duy Thiên, nguyên trinh sát, người làm công tác bản đồ thuộc Trung đoàn 266, Sư đoàn 341-Sông Lam chia sẻ về công tác vẽ bản đồ tác chiến tấn công vào “cánh cửa thép” Xuân Lộc.",neutral,0.600000
5,"Thủy thủ tàu ngầm đọc sách giữa lòng đại dương sâu thẳm — Giữa lòng đại dương sâu thẳm, nơi gần như cách biệt hoàn toàn với thế giới bên ngoài, những thủy thủ tàu ngầm Lữ đoàn 189 duy trì một nét đẹp văn hóa đáng trân trọng.",neutral,0.600000
6,"Tổng Bí thư tri ân các tướng lĩnh, anh hùng lực lượng vũ trang — Tổng Bí thư Tô Lâm nhấn mạnh, đại thắng mùa Xuân 1975 mãi mãi là niềm tự hào, là mốc son chói lọi nhất trong lịch sử của dân tộc, một biểu tượng sáng ngời của chủ nghĩa anh hùng cách mạng.",neutral,0.600000
7,"Lý do hợp nhất Thái Bình với Hưng Yên, lấy tên tỉnh mới là Hưng Yên — Tên gọi “tỉnh Hưng Yên” sau khi hợp nhất Hưng Yên với Thái Bình là phù hợp, vì địa danh này có bề dày lịch sử, văn hiến và truyền thống cách mạng, xuất hiện từ thời vua Minh Mạng năm 1831.",neutral,0.600000
8,"Phu nhân Thủ tướng chia sẻ về Bắc Bling, nghe quan họ với các nhà ngoại giao nữ — Sáng nay, tại tỉnh Bắc Ninh, bà Lê Thị Bích Trân, Phu nhân Thủ tướng Phạm Minh Chính gặp mặt, giao lưu với Nhóm Phụ nữ Cộng đồng ASEAN tại Hà Nội.",neutral,0.600000
9,"'Chúng tôi xung trận không phải để thành anh hùng' — ""Chúng tôi còn sống qua chiến tranh là may mắn và cũng là sứ mệnh, sống để tiếp tục cống hiến cho Tổ quốc thay cả phần những đồng đội đã hy sinh"" - nữ Anh hùng Phan Thị Ngọc Tươi chia sẻ.",neutral,0.600000
